In [ ]:
import transformers

print(transformers.__version__)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


model_name = "google/flan-t5-small"


tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


question = """
Can humans breathe underwater naturally?
"""


inputs = tokenizer(
    question,
    return_tensors="pt"
)


outputs = model.generate(
    **inputs,
    max_length=100
)


answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)


print(answer)

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import pandas as pd
import torch


# Load dataset
dataset = load_dataset(
    "truthfulqa/truthful_qa",
    "generation"
)


data = dataset["validation"]


# Load model
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


results = []


# Run first 100 questions initially
for item in data: #.select(range(100)):

    question = item["question"]

    inputs = tokenizer(
        question,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_length=100
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


    results.append({
        "question": question,
        "generated_answer": answer,
        "best_answer": item["best_answer"]
    })


df = pd.DataFrame(results)


df.to_csv(
    "../results/baseline_results.csv",
    index=False
)


print(df.head())